# Urban Expansion vs Economic Activity
## Unified Data Pipeline & Exploratory Data Analysis

**AC209b Final Project — Junyeob Moon**

---

This notebook consolidates the full data pipeline for studying the relationship between urban expansion (satellite imagery) and economic activity (official surveys) across five U.S. metropolitan areas from 2013–2023.

### Pipeline Overview

| Part | Description |
|------|-------------|
| **1. Setup & Data Loading** | Imports, configuration, inventory of all data assets |
| **2. Satellite Imagery** | Load GeoTIFFs, cloud masking, VIIRS reprojection, band normalization |
| **3. Economic Panel** | Load BEA GDP, BLS employment, Census permits; merge into panel |
| **4. Feature Engineering** | Extract pixel-level statistics from imagery; build modeling features |
| **5. Merging & Alignment** | Join satellite features with economic panel; resolve missingness |
| **6. Normalization** | Apply appropriate scaling for downstream modeling |
| **7. Exploratory Data Analysis** | Distributions, temporal trends, cross-city comparisons, correlations |
| **8. Final Export** | Save modeling-ready datasets |

### Study Area

Five fast-growing U.S. metros selected for diverse geography and growth patterns:
- **Austin, TX** — tech-driven boom
- **Dallas, TX** — large diversified metro
- **Jacksonville, FL** — mid-size coastal growth
- **Nashville, TN** — service/healthcare hub
- **Phoenix, AZ** — arid Sunbelt expansion

### Data Sources

| Source | Variables | Temporal Coverage |
|--------|-----------|-------------------|
| NASA GIBS — MODIS Terra RGB | True-color surface reflectance | 2013–2023 |
| NASA GIBS — VIIRS Day/Night Band | Night-light radiance | 2017–2023 |
| BEA CAGDP9 | Metro GDP (chained 2017\$) | 2013–2023 |
| BLS LAUS | Employment & unemployment | 2013–2023 |
| Census BPS | Building permits | 2013–2023 |

---
## Part 1 — Setup & Data Inventory

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import json
import warnings
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from tqdm import tqdm

# ── Global plot style ────────────────────────────────────────────────────────
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", font_scale=1.15, rc={
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.figsize": (12, 6),
})
PALETTE = sns.color_palette("Set2", 5)
METRO_COLORS = {}  # filled after config

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(fig, name, **kwargs):
    """Save figure to figures/ directory and display."""
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, bbox_inches="tight", facecolor="white", **kwargs)
    print(f"Saved: {path}")

print("Imports complete.")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
METROS = ["austin", "dallas", "jacksonville", "nashville", "phoenix"]
METRO_LABELS = {
    "austin": "Austin, TX", "dallas": "Dallas, TX",
    "jacksonville": "Jacksonville, FL", "nashville": "Nashville, TN",
    "phoenix": "Phoenix, AZ",
}
METRO_COLORS = {m: PALETTE[i] for i, m in enumerate(METROS)}

# Bounding boxes used for tile fetching (min_lon, min_lat, max_lon, max_lat)
BBOXES = {
    "austin":       (-97.94, 30.10, -97.50, 30.52),
    "dallas":       (-97.08, 32.62, -96.55, 33.02),
    "nashville":    (-87.05, 35.96, -86.52, 36.35),
    "phoenix":      (-112.32, 33.29, -111.65, 33.82),
    "jacksonville": (-81.84, 30.10, -81.33, 30.54),
}

# Year ranges
ALL_YEARS   = list(range(2013, 2024))
MODIS_YEARS = list(range(2013, 2024))   # 11 years
VIIRS_YEARS = list(range(2017, 2024))   #  7 years

# Train / val / test split (temporal, no leakage)
TRAIN_YEARS = list(range(2013, 2019))   # 2013–2018
VAL_YEARS   = list(range(2019, 2021))   # 2019–2020
TEST_YEARS  = list(range(2021, 2024))   # 2021–2023

# Paths
IMAGERY_DIR = "data/imagery"
TENSOR_DIR  = "data/tensors"
ECON_DIR    = "data/economic"

print(f"Metros : {METROS}")
print(f"Years  : {ALL_YEARS[0]}–{ALL_YEARS[-1]} ({len(ALL_YEARS)} years)")
print(f"Split  : train {TRAIN_YEARS[0]}–{TRAIN_YEARS[-1]} | val {VAL_YEARS[0]}–{VAL_YEARS[-1]} | test {TEST_YEARS[0]}–{TEST_YEARS[-1]}")

### 1.1 Data Inventory

Verify that all expected satellite imagery and economic data files are present before proceeding.

In [ ]:
# ── Inventory: check all expected GeoTIFFs ───────────────────────────────────
missing, present = [], []
for metro in METROS:
    for year in MODIS_YEARS:
        p = os.path.join(IMAGERY_DIR, metro, "modis_rgb", f"{year}.tif")
        (present if os.path.exists(p) else missing).append(p)
    for year in VIIRS_YEARS:
        p = os.path.join(IMAGERY_DIR, metro, "viirs_night", f"{year}.tif")
        (present if os.path.exists(p) else missing).append(p)

print(f"Satellite imagery: {len(present)} / {len(present)+len(missing)} files present")
if missing:
    print("MISSING FILES:")
    for f in missing:
        print(f"  {f}")
else:
    print("All satellite imagery files present.")

# ── Check economic panel ─────────────────────────────────────────────────────
panel_path = os.path.join(ECON_DIR, "panel.csv")
if os.path.exists(panel_path):
    print(f"\nEconomic panel: {panel_path} exists")
else:
    print(f"\nWARNING: {panel_path} not found — run economic data pipeline first")

---
## Part 2 — Satellite Imagery Preprocessing

The raw satellite imagery was fetched from NASA GIBS (notebook `01_gibs_tile_fetcher_v5`):
- **MODIS Terra True Color** (250 m, zoom 6): 3-band RGB composites, 2013–2023
- **VIIRS Day/Night Band** (500 m, zoom 5): single-band night-light radiance, 2017–2023

Key preprocessing steps:
1. **Cloud masking** — set near-white pixels (all channels > 0.95) to zero
2. **VIIRS reprojection** — resample VIIRS (zoom-5) onto the MODIS grid (zoom-6) via bilinear interpolation
3. **Normalization** — MODIS RGB: divide by 255 to [0,1]; VIIRS: percentile-based min-max to [0,1]
4. **Tensor stacking** — build (T, H, W, 4) arrays per metro (RGB + night-lights)

In [ ]:
# ── Helper: Read GeoTIFF as float32 ──────────────────────────────────────────
def read_tif(path):
    """Read a GeoTIFF → (H, W, bands) float32 in [0,1] + rasterio profile."""
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32) / 255.0   # uint8 → [0,1]
        profile = src.profile.copy()
    return np.transpose(data, (1, 2, 0)), profile      # (bands,H,W) → (H,W,bands)


def cloud_mask(rgb, threshold=0.95):
    """Zero out near-white pixels (likely cloud/snow) in an RGB array."""
    arr = rgb.copy()
    cloud = np.all(arr > threshold, axis=-1)
    arr[cloud] = 0.0
    return arr, cloud.mean() * 100


def resample_viirs_to_modis(viirs_arr, viirs_profile, modis_profile):
    """Bilinear resample VIIRS (zoom-5) onto MODIS grid (zoom-6)."""
    dst_h, dst_w = modis_profile["height"], modis_profile["width"]
    dst = np.zeros((dst_h, dst_w), dtype=np.float32)
    reproject(
        source=viirs_arr[:, :, 0],
        destination=dst,
        src_transform=viirs_profile["transform"],
        src_crs=viirs_profile["crs"],
        dst_transform=modis_profile["transform"],
        dst_crs=modis_profile["crs"],
        resampling=Resampling.bilinear,
    )
    return dst[:, :, np.newaxis]

print("Preprocessing helpers defined.")

### 2.1 Compute Global VIIRS Normalization Statistics

We use the 1st and 99th percentiles of non-zero VIIRS pixel values across all metros and years to define a consistent normalization range. This avoids outlier-driven distortion while preserving the dynamic range of night-light intensity.

In [ ]:
# ── Compute global VIIRS percentile stats ────────────────────────────────────
all_viirs_vals = []
for metro in METROS:
    for year in VIIRS_YEARS:
        p = os.path.join(IMAGERY_DIR, metro, "viirs_night", f"{year}.tif")
        with rasterio.open(p) as src:
            data = src.read(1).astype(np.float32)
        all_viirs_vals.append(data[data > 0])

flat = np.concatenate(all_viirs_vals)
VIIRS_MIN = float(np.percentile(flat, 1))
VIIRS_MAX = float(np.percentile(flat, 99))
print(f"VIIRS normalization range: p1 = {VIIRS_MIN:.1f}, p99 = {VIIRS_MAX:.1f}")
print(f"  (based on {len(flat):,} non-zero pixels across {len(METROS)} metros x {len(VIIRS_YEARS)} years)")
del all_viirs_vals, flat  # free memory

### 2.2 Build Per-Metro Tensor Stacks

For each metro, we assemble a 4-channel tensor of shape `(T, H, W, 4)`:
- Channels 0–2: MODIS RGB (cloud-masked, scaled to [0,1])
- Channel 3: VIIRS night-lights (reprojected to MODIS grid, percentile-normalized to [0,1])
- Years before 2017 have channel 3 = 0 (VIIRS not yet available)

In [ ]:
# ── Build per-metro tensors ───────────────────────────────────────────────────
tensors = {}
profiles = {}
cloud_stats = []  # for EDA later

for metro in METROS:
    # Reference grid from first MODIS file
    ref_path = os.path.join(IMAGERY_DIR, metro, "modis_rgb", f"{ALL_YEARS[0]}.tif")
    with rasterio.open(ref_path) as src:
        ref_profile = src.profile.copy()
        ref_h, ref_w = src.height, src.width

    T = len(ALL_YEARS)
    tensor = np.zeros((T, ref_h, ref_w, 4), dtype=np.float32)

    for t, year in enumerate(tqdm(ALL_YEARS, desc=metro, ncols=80)):
        # MODIS RGB
        modis_path = os.path.join(IMAGERY_DIR, metro, "modis_rgb", f"{year}.tif")
        rgb, _ = read_tif(modis_path)
        rgb, cloud_pct = cloud_mask(rgb)
        cloud_stats.append({"metro": metro, "year": year, "cloud_pct": cloud_pct})
        tensor[t, :, :, :3] = rgb

        # VIIRS
        if year in VIIRS_YEARS:
            viirs_path = os.path.join(IMAGERY_DIR, metro, "viirs_night", f"{year}.tif")
            viirs_raw, viirs_prof = read_tif(viirs_path)
            viirs_norm = np.clip((viirs_raw * 255.0 - VIIRS_MIN) / (VIIRS_MAX - VIIRS_MIN), 0, 1)
            viirs_resampled = resample_viirs_to_modis(viirs_norm, viirs_prof, ref_profile)
            tensor[t, :, :, 3:4] = viirs_resampled

    tensors[metro] = tensor
    profiles[metro] = ref_profile
    print(f"  {metro}: shape={tensor.shape}, range=[{tensor.min():.3f}, {tensor.max():.3f}]")

cloud_df = pd.DataFrame(cloud_stats)
print(f"\nAll tensors built. Cloud masking summary:")
print(cloud_df.groupby("metro")["cloud_pct"].describe().round(2))

### 2.3 Visual Inspection — Satellite Imagery Grid

A visual check of each metro across two time periods: an early year (2015) and a recent year (2022), showing both the true-color MODIS composite and the VIIRS night-light intensity.

In [ ]:
# ── Satellite imagery grid: MODIS RGB + VIIRS for each metro ─────────────────
fig, axes = plt.subplots(5, 4, figsize=(18, 22))

preview_years = [2015, 2022]  # early vs recent
for row, metro in enumerate(METROS):
    tensor = tensors[metro]
    for col_offset, year in enumerate(preview_years):
        t = ALL_YEARS.index(year)
        # MODIS RGB
        ax_rgb = axes[row, col_offset * 2]
        ax_rgb.imshow(tensor[t, :, :, :3])
        ax_rgb.set_title(f"{METRO_LABELS[metro]} — RGB {year}", fontsize=10)
        ax_rgb.axis("off")

        # VIIRS Night
        ax_night = axes[row, col_offset * 2 + 1]
        night = tensor[t, :, :, 3]
        ax_night.imshow(night, cmap="hot", vmin=0, vmax=1)
        label = "(pre-VIIRS)" if year < 2017 else ""
        ax_night.set_title(f"{METRO_LABELS[metro]} — Night {year} {label}", fontsize=10)
        ax_night.axis("off")

plt.suptitle("Satellite Imagery Overview: MODIS RGB & VIIRS Night-Lights", fontsize=15, y=1.01)
plt.tight_layout()
savefig(fig, "01_satellite_imagery_grid.png")
plt.show()

---
## Part 3 — Economic Panel Data

The economic data was assembled from three official U.S. government sources (notebook `02_economic_data_downloader_v6`):

| Source | Variable | Unit |
|--------|----------|------|
| BEA CAGDP9 | `gdp_millions` | Real GDP, millions of chained 2017$ |
| BLS LAUS | `employment_thousands` | Total employment (thousands) |
| BLS LAUS | `unemployment_rate` | Annual average unemployment rate (%) |
| Census BPS | `total_permits` | Total building permits issued |

All indicators are at the metropolitan (CBSA) level, 2013–2023.

In [ ]:
# ── Load economic panel ──────────────────────────────────────────────────────
econ = pd.read_csv(os.path.join(ECON_DIR, "panel.csv"))
econ["year"] = econ["year"].astype(int)

print(f"Shape: {econ.shape}")
print(f"Metros: {sorted(econ['metro'].unique())}")
print(f"Years : {econ['year'].min()}–{econ['year'].max()}")
print(f"\nColumn types:")
print(econ.dtypes)
print(f"\nFirst 5 rows:")
econ.head()

In [ ]:
# ── Missingness check on raw panel ────────────────────────────────────────────
print("Missing values per column:")
print(econ.isnull().sum())
print(f"\nInterpolated rows: {econ['interpolated'].sum()}")
print(f"\nDescriptive statistics:")
econ.describe().round(2)

### 3.1 Economic Panel — Time Series Overview

Four-panel view of all economic indicators over time, with each metro as a separate line. This immediately reveals scale differences across metros, the COVID-19 shock in 2020, and long-term growth trajectories.

In [ ]:
# ── Economic time series panel ────────────────────────────────────────────────
metrics = [
    ("gdp_millions",          "Real GDP ($M, chained 2017$)"),
    ("employment_thousands",  "Employment (thousands)"),
    ("unemployment_rate",     "Unemployment Rate (%)"),
    ("total_permits",         "Building Permits Issued"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (col, title) in zip(axes.flat, metrics):
    for metro in METROS:
        subset = econ[econ["metro"] == metro]
        ax.plot(subset["year"], subset[col], marker="o", markersize=4,
                label=METRO_LABELS[metro], color=METRO_COLORS[metro], linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    # Highlight COVID year
    ax.axvspan(2019.5, 2020.5, alpha=0.08, color="red", label="_covid")

axes[0, 1].legend(loc="upper left", fontsize=9, framealpha=0.9)
plt.suptitle("Economic Indicators by Metro Area (2013–2023)", fontsize=14, y=1.02)
plt.tight_layout()
savefig(fig, "02_economic_timeseries.png")
plt.show()

---
## Part 4 — Feature Engineering from Satellite Imagery

We extract per-metro, per-year summary statistics from the imagery tensors to create tabular features that can be joined with the economic panel. These capture:

1. **MODIS RGB statistics** — mean/std of each band, brightness (proxy for surface reflectance)
2. **VIIRS night-light statistics** — mean intensity, lit-pixel fraction, intensity percentiles
3. **Temporal change features** — year-over-year changes in brightness and night-light intensity
4. **Spatial features** — urban concentration metrics from night-light spatial distribution

In [ ]:
# ── Extract satellite-derived features per (metro, year) ─────────────────────
sat_features = []

for metro in METROS:
    tensor = tensors[metro]
    T, H, W, C = tensor.shape
    n_pixels = H * W

    for t, year in enumerate(ALL_YEARS):
        row = {"metro": metro, "year": year}

        # ── MODIS RGB features (channels 0-2) ───────────────────────────
        rgb = tensor[t, :, :, :3]
        for ch, name in enumerate(["red", "green", "blue"]):
            row[f"modis_{name}_mean"] = float(rgb[:, :, ch].mean())
            row[f"modis_{name}_std"]  = float(rgb[:, :, ch].std())

        # Brightness = mean across RGB channels
        brightness = rgb.mean(axis=-1)
        row["modis_brightness_mean"] = float(brightness.mean())
        row["modis_brightness_std"]  = float(brightness.std())

        # NDVI proxy: (Green - Red) / (Green + Red + 1e-8)
        # Higher values suggest more vegetation (less urban)
        ndvi_proxy = (rgb[:, :, 1] - rgb[:, :, 0]) / (rgb[:, :, 1] + rgb[:, :, 0] + 1e-8)
        row["modis_ndvi_proxy_mean"] = float(ndvi_proxy.mean())

        # Dark pixel fraction (brightness < 0.15) — water, shadow, or deep vegetation
        row["modis_dark_frac"] = float((brightness < 0.15).mean())

        # ── VIIRS night-light features (channel 3) ──────────────────────
        night = tensor[t, :, :, 3]
        has_viirs = year >= 2017

        row["viirs_available"] = has_viirs
        row["viirs_mean"]      = float(night.mean())
        row["viirs_std"]       = float(night.std())
        row["viirs_median"]    = float(np.median(night))
        row["viirs_p90"]       = float(np.percentile(night, 90))
        row["viirs_max"]       = float(night.max())

        # Lit pixel fraction (above noise threshold)
        if has_viirs:
            row["viirs_lit_frac"]   = float((night > 0.1).mean())
            row["viirs_bright_frac"] = float((night > 0.5).mean())
            # Gini coefficient of night-light (spatial concentration)
            flat = np.sort(night.ravel())
            n = len(flat)
            index = np.arange(1, n + 1)
            row["viirs_gini"] = float((2 * (index * flat).sum() / (n * flat.sum() + 1e-8)) - (n + 1) / n)
        else:
            row["viirs_lit_frac"]    = np.nan
            row["viirs_bright_frac"] = np.nan
            row["viirs_gini"]        = np.nan

        sat_features.append(row)

sat_df = pd.DataFrame(sat_features)
print(f"Satellite features: {sat_df.shape}")
print(f"Columns: {list(sat_df.columns)}")
sat_df.head()

### 4.1 Temporal Change Features

Year-over-year deltas capture the *dynamics* of urban expansion — how quickly night-lights are growing, whether surface brightness is shifting — rather than just static levels.

In [ ]:
# ── Compute year-over-year changes ────────────────────────────────────────────
change_cols = [
    "modis_brightness_mean", "modis_ndvi_proxy_mean", "modis_dark_frac",
    "viirs_mean", "viirs_lit_frac", "viirs_bright_frac",
]

sat_df = sat_df.sort_values(["metro", "year"]).reset_index(drop=True)

for col in change_cols:
    sat_df[f"{col}_delta"] = sat_df.groupby("metro")[col].diff()

# Growth rates (percentage change) for key indicators
for col in ["modis_brightness_mean", "viirs_mean"]:
    sat_df[f"{col}_growth"] = sat_df.groupby("metro")[col].pct_change() * 100

# Also add lag features for economic modeling (previous year's satellite values)
for col in ["viirs_mean", "viirs_lit_frac", "modis_brightness_mean"]:
    sat_df[f"{col}_lag1"] = sat_df.groupby("metro")[col].shift(1)

print(f"Satellite features after temporal engineering: {sat_df.shape}")
print(f"New delta/growth/lag columns: {[c for c in sat_df.columns if 'delta' in c or 'growth' in c or 'lag' in c]}")

---
## Part 5 — Merging, Alignment & Missingness Resolution

We join the satellite-derived features with the economic panel on `(metro, year)`, then systematically check for and resolve any alignment or missingness issues.

In [ ]:
# ── Merge satellite features with economic panel ─────────────────────────────
panel = econ.merge(sat_df, on=["metro", "year"], how="outer", indicator=True)

print("Merge diagnostics:")
print(panel["_merge"].value_counts())
print(f"\nFinal merged panel: {panel.shape[0]} rows x {panel.shape[1]} columns")

# Verify perfect alignment: 5 metros x 11 years = 55 rows
assert panel.shape[0] == len(METROS) * len(ALL_YEARS), "Row count mismatch!"
assert (panel["_merge"] == "both").all(), "Some rows did not match!"
panel = panel.drop(columns=["_merge"])

print("All (metro, year) pairs align perfectly between satellite and economic data.")

### 5.1 Missingness Audit

Systematic review of missing values. VIIRS-derived features are structurally missing for 2013–2016 (pre-VIIRS). Temporal change features have missing first-year values by construction.

In [ ]:
# ── Missingness heatmap ──────────────────────────────────────────────────────
missing = panel.isnull()
cols_with_missing = missing.columns[missing.any()]

if len(cols_with_missing) > 0:
    # Build a tidy label for each row
    labels = panel["metro"].str.title() + " " + panel["year"].astype(str)

    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(
        missing[cols_with_missing].astype(int).values,
        xticklabels=[c.replace("_", " ") for c in cols_with_missing],
        yticklabels=labels,
        cmap=["#f0f0f0", "#e74c3c"],
        cbar_kws={"label": "Missing", "ticks": [0, 1]},
        linewidths=0.3, linecolor="#ddd",
        ax=ax,
    )
    ax.set_title("Missingness Map (red = missing)")
    ax.set_ylabel("Metro / Year")
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    savefig(fig, "03_missingness_heatmap.png")
    plt.show()

    print(f"\nColumns with missing values: {len(cols_with_missing)}")
    print(missing[cols_with_missing].sum().to_string())
else:
    print("No missing values in any column.")

### 5.2 Missingness Resolution Strategy

| Source of missingness | Affected columns | Resolution |
|---|---|---|
| VIIRS not available pre-2017 | `viirs_lit_frac`, `viirs_bright_frac`, `viirs_gini` | Mark as structurally missing; use `viirs_available` flag in models |
| First-year delta/lag features | `*_delta`, `*_growth`, `*_lag1` | Structural — first year has no prior; leave as NaN for models that handle it, or drop first year |
| Economic interpolation | None expected | Already handled in data download step |

In [ ]:
# ── Add economic temporal features ────────────────────────────────────────────
panel = panel.sort_values(["metro", "year"]).reset_index(drop=True)

# Year-over-year growth for economic indicators
for col in ["gdp_millions", "employment_thousands", "total_permits"]:
    panel[f"{col}_growth"] = panel.groupby("metro")[col].pct_change() * 100

# Lag-1 economic features
for col in ["gdp_millions", "employment_thousands", "unemployment_rate", "total_permits"]:
    panel[f"{col}_lag1"] = panel.groupby("metro")[col].shift(1)

# GDP per employee (productivity proxy)
panel["gdp_per_employee"] = panel["gdp_millions"] / panel["employment_thousands"]

# Permits per 1000 employees (construction intensity)
panel["permits_per_1k_emp"] = panel["total_permits"] / panel["employment_thousands"]

# Add train/val/test split label
panel["split"] = panel["year"].apply(
    lambda y: "train" if y in TRAIN_YEARS else ("val" if y in VAL_YEARS else "test")
)

print(f"Panel after feature engineering: {panel.shape}")
print(f"\nSplit distribution:")
print(panel["split"].value_counts().sort_index())

---
## Part 6 — Normalization

We apply z-score standardization (mean=0, std=1) to all numeric features using **training-set statistics only** to prevent data leakage. The original (un-normalized) panel is preserved for interpretable EDA, while the normalized version is saved for modeling.

In [ ]:
# ── Z-score normalization using training-set statistics ───────────────────────
exclude_cols = {"metro", "year", "split", "interpolated", "viirs_available"}
numeric_cols = [c for c in panel.columns
                if c not in exclude_cols and panel[c].dtype in [np.float64, np.float32, np.int64]]

train_mask = panel["split"] == "train"
train_means = panel.loc[train_mask, numeric_cols].mean()
train_stds  = panel.loc[train_mask, numeric_cols].std().replace(0, 1)  # avoid div-by-zero

panel_norm = panel.copy()
panel_norm[numeric_cols] = (panel[numeric_cols] - train_means) / train_stds

print(f"Normalized {len(numeric_cols)} numeric columns using training-set statistics.")
print(f"Training set: {train_mask.sum()} rows | Val: {(panel['split']=='val').sum()} | Test: {(panel['split']=='test').sum()}")
print(f"\nNormalized feature ranges (all splits):")
panel_norm[numeric_cols].describe().round(2).loc[["mean", "std", "min", "max"]]

---
## Part 7 — Comprehensive Exploratory Data Analysis

### 7.1 Feature Distributions

Distributions of key satellite-derived and economic features across the full panel. Understanding the distributional shape informs modeling choices (e.g., whether log transforms are needed, presence of outliers).

In [ ]:
# ── Feature distributions: violin plots by metro ─────────────────────────────
dist_features = [
    ("gdp_millions", "GDP ($ Millions)"),
    ("employment_thousands", "Employment (Thousands)"),
    ("total_permits", "Building Permits"),
    ("viirs_mean", "VIIRS Mean Intensity"),
    ("modis_brightness_mean", "MODIS Brightness"),
    ("viirs_lit_frac", "VIIRS Lit-Pixel Fraction"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, (col, label) in zip(axes.flat, dist_features):
    data = panel.dropna(subset=[col])
    parts = ax.violinplot(
        [data.loc[data["metro"] == m, col].values for m in METROS],
        positions=range(len(METROS)), showmeans=True, showmedians=True,
    )
    for i, pc in enumerate(parts["bodies"]):
        pc.set_facecolor(METRO_COLORS[METROS[i]])
        pc.set_alpha(0.7)
    ax.set_xticks(range(len(METROS)))
    ax.set_xticklabels([m.title() for m in METROS], fontsize=9)
    ax.set_title(label)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Feature Distributions by Metro Area", fontsize=14, y=1.02)
plt.tight_layout()
savefig(fig, "04_feature_distributions.png")
plt.show()

### 7.2 Satellite Feature Temporal Trends

How satellite-derived features evolve over time. Night-light intensity should track economic growth; MODIS brightness may shift with land-use change.

In [ ]:
# ── Satellite feature trends over time ────────────────────────────────────────
sat_trends = [
    ("viirs_mean",             "VIIRS Mean Intensity"),
    ("viirs_lit_frac",         "VIIRS Lit-Pixel Fraction"),
    ("viirs_gini",             "VIIRS Gini (Spatial Concentration)"),
    ("modis_brightness_mean",  "MODIS Mean Brightness"),
    ("modis_ndvi_proxy_mean",  "MODIS NDVI Proxy"),
    ("modis_dark_frac",        "MODIS Dark-Pixel Fraction"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (col, title) in zip(axes.flat, sat_trends):
    for metro in METROS:
        sub = panel[(panel["metro"] == metro) & panel[col].notna()]
        ax.plot(sub["year"], sub[col], marker="o", markersize=4,
                label=METRO_LABELS[metro], color=METRO_COLORS[metro], linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(alpha=0.3)
    # Shade VIIRS unavailability
    if "viirs" in col:
        ax.axvspan(2012.5, 2016.5, alpha=0.06, color="gray")
        ax.text(2014.5, ax.get_ylim()[1] * 0.95, "No VIIRS", fontsize=7,
                ha="center", color="gray", style="italic")

axes[0, 2].legend(loc="best", fontsize=8, framealpha=0.9)
plt.suptitle("Satellite-Derived Feature Trends (2013–2023)", fontsize=14, y=1.02)
plt.tight_layout()
savefig(fig, "05_satellite_feature_trends.png")
plt.show()

### 7.3 Correlation Analysis

**Interpretation guide:** We expect night-light intensity to correlate positively with GDP and employment, as brighter/more extensive night-lights indicate greater economic activity. Building permits (a forward-looking indicator) may correlate with satellite-derived growth rates.

In [ ]:
# ── Correlation heatmap: satellite features vs economic indicators ────────────
corr_sat = [
    "viirs_mean", "viirs_lit_frac", "viirs_bright_frac", "viirs_gini",
    "modis_brightness_mean", "modis_ndvi_proxy_mean", "modis_dark_frac",
]
corr_econ = [
    "gdp_millions", "employment_thousands", "unemployment_rate", "total_permits",
    "gdp_per_employee", "permits_per_1k_emp",
]

# Use only rows with VIIRS data for meaningful correlations
viirs_panel = panel[panel["viirs_available"] == True].copy()
corr_matrix = viirs_panel[corr_sat + corr_econ].corr()

# Extract the cross-correlation block
cross_corr = corr_matrix.loc[corr_sat, corr_econ]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    cross_corr, annot=True, fmt=".2f", center=0,
    cmap="RdBu_r", vmin=-1, vmax=1,
    linewidths=0.5, linecolor="white",
    xticklabels=[c.replace("_", "\n") for c in corr_econ],
    yticklabels=[c.replace("_", "\n") for c in corr_sat],
    ax=ax,
)
ax.set_title("Satellite vs Economic Feature Correlations (2017–2023, all metros)")
plt.tight_layout()
savefig(fig, "06_cross_correlation_heatmap.png")
plt.show()

### 7.4 Scatter Plots — Key Satellite-Economic Relationships

Direct visual evidence for the core hypothesis: do night-light changes track economic activity?

In [ ]:
# ── Scatter plots: satellite vs economic, colored by metro ───────────────────
scatter_pairs = [
    ("viirs_mean",       "gdp_millions",          "VIIRS Mean Intensity", "GDP ($M)"),
    ("viirs_lit_frac",   "employment_thousands",   "VIIRS Lit-Pixel Fraction", "Employment (K)"),
    ("viirs_bright_frac","total_permits",           "VIIRS Bright Fraction", "Building Permits"),
    ("modis_brightness_mean", "gdp_per_employee",  "MODIS Brightness", "GDP per Employee ($K)"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, (x, y, xlabel, ylabel) in zip(axes.flat, scatter_pairs):
    for metro in METROS:
        sub = viirs_panel[viirs_panel["metro"] == metro]
        ax.scatter(sub[x], sub[y], s=50, alpha=0.8,
                   label=METRO_LABELS[metro], color=METRO_COLORS[metro],
                   edgecolors="white", linewidth=0.5)
    # Add overall trend line
    valid = viirs_panel[[x, y]].dropna()
    if len(valid) > 2:
        slope, intercept, r, p, se = stats.linregress(valid[x], valid[y])
        x_line = np.linspace(valid[x].min(), valid[x].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, "--", color="gray", linewidth=1.5, alpha=0.7)
        ax.text(0.05, 0.95, f"r = {r:.2f}\np = {p:.3f}", transform=ax.transAxes,
                fontsize=9, va="top", bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)

axes[0, 0].legend(loc="lower right", fontsize=8, framealpha=0.9)
plt.suptitle("Satellite Features vs Economic Indicators (2017–2023)", fontsize=14, y=1.02)
plt.tight_layout()
savefig(fig, "07_satellite_vs_economic_scatter.png")
plt.show()

### 7.5 Within-Metro Correlations

The pooled correlations above mix between-metro variation (Dallas is bigger than Jacksonville) with within-metro temporal variation. Here we compute per-metro correlations to isolate the temporal signal: does *change* in night-lights within a city track *change* in its economy?

In [ ]:
# ── Per-metro temporal correlations ───────────────────────────────────────────
within_pairs = [
    ("viirs_mean", "gdp_millions"),
    ("viirs_mean", "employment_thousands"),
    ("viirs_lit_frac", "total_permits"),
    ("viirs_mean", "unemployment_rate"),
]

results = []
for metro in METROS:
    sub = viirs_panel[viirs_panel["metro"] == metro]
    for sat_col, econ_col in within_pairs:
        valid = sub[[sat_col, econ_col]].dropna()
        if len(valid) >= 4:
            r, p = stats.pearsonr(valid[sat_col], valid[econ_col])
            results.append({"metro": metro, "sat": sat_col, "econ": econ_col, "r": r, "p": p, "n": len(valid)})

within_corr = pd.DataFrame(results)
pivot = within_corr.pivot_table(index="metro", columns=["sat", "econ"], values="r")

fig, ax = plt.subplots(figsize=(12, 4))
# Reshape for heatmap
display_df = within_corr.pivot(index="metro", columns="econ", values="r")
display_df.index = [METRO_LABELS[m] for m in display_df.index]
display_df.columns = [c.replace("_", " ").title() for c in display_df.columns]

sns.heatmap(display_df, annot=True, fmt=".2f", center=0, cmap="RdBu_r",
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Within-Metro Temporal Correlations: VIIRS Mean vs Economic Indicators (2017–2023)")
plt.tight_layout()
savefig(fig, "08_within_metro_correlations.png")
plt.show()

### 7.6 Night-Light Change Maps

Spatial visualization of how night-light intensity changed between 2017 (earliest VIIRS year) and 2023 (latest year). Red/warm colors indicate areas of increasing night-light, blue indicates decreasing — a direct proxy for spatial patterns of urban expansion.

In [ ]:
# ── Night-light change maps: 2017 → 2023 ─────────────────────────────────────
t_start = ALL_YEARS.index(2017)
t_end   = ALL_YEARS.index(2023)

fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
for ax, metro in zip(axes, METROS):
    night_start = tensors[metro][t_start, :, :, 3]
    night_end   = tensors[metro][t_end,   :, :, 3]
    change = night_end - night_start

    vmax = max(abs(change.min()), abs(change.max()), 0.3)
    im = ax.imshow(change, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(f"{METRO_LABELS[metro]}\n2017 → 2023", fontsize=10)
    ax.axis("off")

cbar = fig.colorbar(im, ax=axes, orientation="horizontal", fraction=0.04, pad=0.12)
cbar.set_label("Night-Light Intensity Change (2023 minus 2017)")
plt.suptitle("Spatial Patterns of Night-Light Change", fontsize=14, y=1.05)
plt.tight_layout()
savefig(fig, "09_nightlight_change_maps.png")
plt.show()

### 7.7 COVID-19 Impact Analysis

The year 2020 provides a natural experiment: economic shutdowns should be visible in both economic indicators (employment drop, unemployment spike) and potentially in night-light patterns (reduced commercial activity). We examine whether the satellite data captures this shock.

In [ ]:
# ── COVID impact: index 2019=100 for key indicators ──────────────────────────
covid_cols = ["gdp_millions", "employment_thousands", "total_permits", "viirs_mean"]
covid_labels = ["GDP", "Employment", "Building Permits", "VIIRS Night-Light"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col, label in zip(axes.flat, covid_cols, covid_labels):
    for metro in METROS:
        sub = panel[panel["metro"] == metro].set_index("year")
        base = sub.loc[2019, col]
        if base > 0:
            indexed = sub[col] / base * 100
            ax.plot(indexed.index, indexed.values, marker="o", markersize=4,
                    label=METRO_LABELS[metro], color=METRO_COLORS[metro], linewidth=2)
    ax.axhline(100, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.axvspan(2019.5, 2020.5, alpha=0.1, color="red")
    ax.set_title(f"{label} (2019 = 100)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Index")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(alpha=0.3)

axes[0, 0].legend(loc="upper left", fontsize=8, framealpha=0.9)
plt.suptitle("COVID-19 Impact: Indexed to 2019 Baseline", fontsize=14, y=1.02)
plt.tight_layout()
savefig(fig, "10_covid_impact_indexed.png")
plt.show()

### 7.8 Growth Rate Comparison

Annualized growth rates reveal which metros are expanding fastest and whether satellite-derived growth tracks economic growth rates.

In [ ]:
# ── Compound annual growth rates (CAGR) ──────────────────────────────────────
def cagr(start_val, end_val, n_years):
    if start_val <= 0:
        return np.nan
    return ((end_val / start_val) ** (1 / n_years) - 1) * 100

growth_data = []
for metro in METROS:
    sub = panel[panel["metro"] == metro].set_index("year")
    row = {"metro": METRO_LABELS[metro]}
    # Full period CAGR
    for col, label in [("gdp_millions", "GDP"), ("employment_thousands", "Employment"),
                       ("total_permits", "Permits")]:
        row[f"{label}\n2013–2023"] = cagr(sub.loc[2013, col], sub.loc[2023, col], 10)
    # VIIRS period
    for col, label in [("viirs_mean", "VIIRS\nIntensity")]:
        row[f"{label}\n2017–2023"] = cagr(sub.loc[2017, col], sub.loc[2023, col], 6)
    growth_data.append(row)

growth_df = pd.DataFrame(growth_data).set_index("metro")

fig, ax = plt.subplots(figsize=(12, 5))
growth_df.plot(kind="bar", ax=ax, width=0.75, edgecolor="white", linewidth=0.5)
ax.set_ylabel("CAGR (%)")
ax.set_title("Compound Annual Growth Rates by Metro")
ax.axhline(0, color="black", linewidth=0.5)
ax.legend(title="Indicator", fontsize=9, title_fontsize=9, loc="upper right")
plt.xticks(rotation=0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
savefig(fig, "11_growth_rates_cagr.png")
plt.show()

print(growth_df.round(2).to_string())

### 7.9 Pixel-Level Distribution Analysis

Distribution of pixel values within each band helps identify data quality issues (e.g., saturation, unusual distributions) and informs choices about normalization and thresholding.

In [ ]:
# ── Pixel-level histograms per band, one metro ──────────────────────────────
sample_metro = "austin"
sample_years = [2017, 2020, 2023]
channel_names = ["MODIS Red", "MODIS Green", "MODIS Blue", "VIIRS Night"]
channel_colors = ["#e74c3c", "#27ae60", "#3498db", "#f39c12"]

fig, axes = plt.subplots(len(sample_years), 4, figsize=(18, 3.5 * len(sample_years)))
for row, year in enumerate(sample_years):
    t = ALL_YEARS.index(year)
    for ch in range(4):
        ax = axes[row, ch]
        vals = tensors[sample_metro][t, :, :, ch].ravel()
        ax.hist(vals, bins=80, color=channel_colors[ch], alpha=0.75,
                edgecolor="white", linewidth=0.3, density=True)
        ax.set_title(f"{channel_names[ch]} — {year}", fontsize=10)
        ax.set_xlabel("Pixel Value")
        if ch == 0:
            ax.set_ylabel("Density")
        # Stats annotation
        ax.text(0.95, 0.95, f"mean={vals.mean():.3f}\nstd={vals.std():.3f}",
                transform=ax.transAxes, fontsize=8, va="top", ha="right",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
        ax.grid(alpha=0.2)

plt.suptitle(f"Pixel Value Distributions — {METRO_LABELS[sample_metro]}", fontsize=14, y=1.02)
plt.tight_layout()
savefig(fig, "12_pixel_distributions.png")
plt.show()

### 7.10 Cross-City Comparison — Pairplot of Key Features

A pairplot of the most important features colored by metro reveals whether cities occupy distinct regions of feature space and whether satellite features separate them.

In [ ]:
# ── Pairplot of key features ──────────────────────────────────────────────────
pair_cols = ["viirs_mean", "modis_brightness_mean", "gdp_millions", "total_permits"]
pair_data = viirs_panel[["metro"] + pair_cols].dropna().copy()
pair_data["metro"] = pair_data["metro"].map(METRO_LABELS)

g = sns.pairplot(
    pair_data, hue="metro", diag_kind="kde",
    palette={METRO_LABELS[m]: METRO_COLORS[m] for m in METROS},
    plot_kws={"s": 50, "alpha": 0.8, "edgecolor": "white", "linewidth": 0.5},
    diag_kws={"alpha": 0.6},
    height=2.8,
)
g.figure.suptitle("Pairplot: Satellite & Economic Features by Metro (2017–2023)", y=1.02, fontsize=13)
savefig(g.figure, "13_pairplot_key_features.png")
plt.show()

### 7.11 Train/Val/Test Split Visualization

Confirming the temporal split is clean and visualizing how each subset spans the data range.

In [ ]:
# ── Train/val/test split visualization ────────────────────────────────────────
split_colors = {"train": "#2ecc71", "val": "#f39c12", "test": "#e74c3c"}

fig, ax = plt.subplots(figsize=(14, 4))
for metro_idx, metro in enumerate(METROS):
    sub = panel[panel["metro"] == metro]
    for _, row in sub.iterrows():
        ax.barh(metro_idx, 0.8, left=row["year"] - 0.4,
                color=split_colors[row["split"]], edgecolor="white", linewidth=0.5)

ax.set_yticks(range(len(METROS)))
ax.set_yticklabels([METRO_LABELS[m] for m in METROS])
ax.set_xlabel("Year")
ax.set_title("Temporal Train / Validation / Test Split")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=f"{s.title()} ({len(panel[panel['split']==s])//5} years)")
                   for s, c in split_colors.items()]
ax.legend(handles=legend_elements, loc="upper right", fontsize=10)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
savefig(fig, "14_train_val_test_split.png")
plt.show()

---
## Part 8 — Final Dataset Export

We save three outputs:
1. **`data/modeling/panel_features.csv`** — full merged panel with all features (un-normalized), for interpretable analysis
2. **`data/modeling/panel_normalized.csv`** — z-score normalized version, for modeling
3. **Updated tensor stacks** — `data/tensors/{metro}_stack.npz` with consistent preprocessing
4. **`data/modeling/normalization_stats.json`** — training-set means and stds for reproducibility

In [ ]:
# ── Save final datasets ──────────────────────────────────────────────────────
MODELING_DIR = "data/modeling"
os.makedirs(MODELING_DIR, exist_ok=True)

# 1. Un-normalized panel
panel_path = os.path.join(MODELING_DIR, "panel_features.csv")
panel.to_csv(panel_path, index=False)
print(f"Saved: {panel_path}  ({panel.shape[0]} rows x {panel.shape[1]} cols)")

# 2. Normalized panel
norm_path = os.path.join(MODELING_DIR, "panel_normalized.csv")
panel_norm.to_csv(norm_path, index=False)
print(f"Saved: {norm_path}  ({panel_norm.shape[0]} rows x {panel_norm.shape[1]} cols)")

# 3. Normalization statistics (for applying to new data)
norm_stats = {
    "train_means": train_means.to_dict(),
    "train_stds":  train_stds.to_dict(),
    "viirs_min":   VIIRS_MIN,
    "viirs_max":   VIIRS_MAX,
    "train_years": TRAIN_YEARS,
    "val_years":   VAL_YEARS,
    "test_years":  TEST_YEARS,
}
stats_path = os.path.join(MODELING_DIR, "normalization_stats.json")
with open(stats_path, "w") as f:
    json.dump(norm_stats, f, indent=2, default=str)
print(f"Saved: {stats_path}")

# 4. Save updated tensors
train_idx = np.array([ALL_YEARS.index(y) for y in TRAIN_YEARS])
val_idx   = np.array([ALL_YEARS.index(y) for y in VAL_YEARS])
test_idx  = np.array([ALL_YEARS.index(y) for y in TEST_YEARS])

for metro, tensor in tensors.items():
    out_path = os.path.join(TENSOR_DIR, f"{metro}_stack.npz")
    np.savez_compressed(
        out_path,
        tensor=tensor,
        years=np.array(ALL_YEARS),
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
    )
    size_mb = os.path.getsize(out_path) / 1e6
    print(f"Saved: {out_path}  ({size_mb:.1f} MB)")

# Update metadata
meta = {
    "metros": METROS,
    "all_years": ALL_YEARS,
    "train_years": TRAIN_YEARS,
    "val_years": VAL_YEARS,
    "test_years": TEST_YEARS,
    "n_channels": 4,
    "channels": ["modis_R", "modis_G", "modis_B", "viirs_night"],
    "viirs_min": VIIRS_MIN,
    "viirs_max": VIIRS_MAX,
    "viirs_years": VIIRS_YEARS,
    "note_viirs": "channel 3 = 0 for years before 2017 (VIIRS not available)",
}
meta_path = os.path.join(TENSOR_DIR, "metadata.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"Saved: {meta_path}")

### 8.1 Final Data Summary

In [ ]:
# ── Final summary ────────────────────────────────────────────────────────────
print("=" * 70)
print("DATASET SUMMARY")
print("=" * 70)
print(f"\nPanel dataset:")
print(f"  Rows    : {panel.shape[0]} ({len(METROS)} metros x {len(ALL_YEARS)} years)")
print(f"  Columns : {panel.shape[1]}")
print(f"  Features: {len(numeric_cols)} numeric + metro + year + split")
print(f"  Missing : {panel[numeric_cols].isnull().sum().sum()} total NaN values")
print(f"           ({panel[numeric_cols].isnull().mean().mean()*100:.1f}% overall missingness rate)")

print(f"\nTensor stacks:")
for metro in METROS:
    t = tensors[metro]
    print(f"  {metro:15s}: {t.shape}  ({t.nbytes/1e6:.1f} MB)")

print(f"\nTemporal split:")
print(f"  Train : {TRAIN_YEARS[0]}–{TRAIN_YEARS[-1]} ({len(TRAIN_YEARS)} years, {len(TRAIN_YEARS)*len(METROS)} obs)")
print(f"  Val   : {VAL_YEARS[0]}–{VAL_YEARS[-1]} ({len(VAL_YEARS)} years, {len(VAL_YEARS)*len(METROS)} obs)")
print(f"  Test  : {TEST_YEARS[0]}–{TEST_YEARS[-1]} ({len(TEST_YEARS)} years, {len(TEST_YEARS)*len(METROS)} obs)")

print(f"\nOutput files:")
for root, dirs, files in os.walk("data/modeling"):
    for f in files:
        fp = os.path.join(root, f)
        print(f"  {fp} ({os.path.getsize(fp)/1e3:.1f} KB)")

print(f"\nFigures saved to: {FIG_DIR}/")
for f in sorted(os.listdir(FIG_DIR)):
    print(f"  {f}")

print("\n" + "=" * 70)
print("Pipeline complete. Data is ready for downstream modeling.")
print("=" * 70)

---
## Output Structure

```
data/
├── imagery/                          # Raw satellite GeoTIFFs (from notebook 01)
│   └── {metro}/{modis_rgb,viirs_night}/{year}.tif
├── economic/
│   └── panel.csv                     # Raw economic panel (from notebook 02)
├── tensors/                          # Preprocessed image tensors
│   ├── {metro}_stack.npz             # (T, H, W, 4) float32 arrays
│   └── metadata.json                 # Channel definitions, year ranges
└── modeling/                         # Final modeling-ready outputs
    ├── panel_features.csv            # Merged satellite + economic features (un-normalized)
    ├── panel_normalized.csv          # Z-score normalized version
    └── normalization_stats.json      # Training-set means/stds for reproducibility

figures/                              # All EDA visualizations
├── 01_satellite_imagery_grid.png
├── 02_economic_timeseries.png
├── 03_missingness_heatmap.png
├── 04_feature_distributions.png
├── 05_satellite_feature_trends.png
├── 06_cross_correlation_heatmap.png
├── 07_satellite_vs_economic_scatter.png
├── 08_within_metro_correlations.png
├── 09_nightlight_change_maps.png
├── 10_covid_impact_indexed.png
├── 11_growth_rates_cagr.png
├── 12_pixel_distributions.png
├── 13_pairplot_key_features.png
└── 14_train_val_test_split.png
```

**Next step: `04_unet_segmentation.ipynb`** — CNN-based urban footprint extraction using the tensor stacks, with the economic panel as prediction targets.